In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# V1

In [ ]:
pip install git+https://github.com/csebuetnlp/normalizer

# Class_balanced_loss

# Import all required libraries at the top
from sklearn.metrics import average_precision_score, classification_report
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig, 
    AutoModelForSequenceClassification, 
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score
from tqdm import tqdm
from normalizer import normalize

# Add these imports at the top of your existing imports
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

# Add these custom loss classes after your imports and before the run function
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class ClassBalancedLoss(nn.Module):
    def __init__(self, samples_per_class, beta=0.9999, reduction='mean'):
        super(ClassBalancedLoss, self).__init__()
        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / np.array(effective_num)
        weights = weights / np.sum(weights) * len(weights)
        self.weights = torch.tensor(weights, dtype=torch.float32)
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        weights = self.weights.to(inputs.device)
        ce_loss = F.cross_entropy(inputs, targets, weight=weights, reduction=self.reduction)
        return ce_loss

def run(seed, model_name, lr, train_loc, val_loc, test_loc, 
        use_focal_loss=False, use_class_balanced_loss=False, use_cost_sensitive_weighting=False,
        focal_alpha=1.0, focal_gamma=2.0, cb_beta=0.9999):
    # Set seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Load data
    train = pd.read_csv(train_loc)
    test = pd.read_csv(test_loc)
    val = pd.read_csv(val_loc)
    
    # Hyperparameters
    batch_size = 16
    max_tokens = 512
    epochs = 200
    best_roc_auc = 0.0
    min_delta = 0.0001
    early_stopping_count = 0
    early_stopping_patience = 3
    gradient_accumulation_steps = 2
    weight_decay = 0.01
    pruning_ratio = 0.3
    dropout_prob = 0.2
    
    # Load model configuration and tokenizer
    config = AutoConfig.from_pretrained(model_name, num_labels=2)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Handle tokenizers without pad_token (like XLM-RoBERTa)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load the pre-trained model with the specified configuration
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    class LosDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels
    
        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx])
            return item
    
        def __len__(self):
            return len(self.labels)

    # Normalize text data
    train['text'] = train['text'].apply(normalize)
    val['text'] = val['text'].apply(normalize)
    test['text'] = test['text'].apply(normalize)

    train_labels = train['los_label'].values
    unique_classes = np.unique(train_labels)
    samples_per_class = np.bincount(train_labels)
    
    # Compute class weights for cost-sensitive learning
    class_weights = compute_class_weight('balanced', classes=unique_classes, y=train_labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
    
    # Initialize loss functions
    if use_focal_loss:
        criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
        print(f"Using Focal Loss with alpha={focal_alpha}, gamma={focal_gamma}")
    elif use_class_balanced_loss:
        criterion = ClassBalancedLoss(samples_per_class, beta=cb_beta)
        print(f"Using Class Balanced Loss with beta={cb_beta}")
    elif use_cost_sensitive_weighting:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
        print(f"Using Cost Sensitive Weighting with weights: {class_weights}")
    else:
        criterion = nn.CrossEntropyLoss()
        print("Using standard Cross Entropy Loss")

    criterion = criterion.to(device)

    # Tokenize data
    train_encodings = tokenizer(train['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)
    val_encodings = tokenizer(val['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)
    test_encodings = tokenizer(test['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)

    # Create datasets
    train_dataset = LosDataset(train_encodings, train['los_label'].tolist())
    val_dataset = LosDataset(val_encodings, val['los_label'].tolist())
    test_dataset = LosDataset(test_encodings, test['los_label'].tolist())

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    # Setup optimizer and scheduler
    warmup_ratio = 0.1
    total_training_steps = (epochs * len(train_loader)) // gradient_accumulation_steps
    num_warmup_steps = int(warmup_ratio * total_training_steps)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_training_steps
    )

    
    scaler = torch.cuda.amp.GradScaler()
    # Training loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0
    
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch+1}")):
            if step % gradient_accumulation_steps == 0:
                optimizer.zero_grad()
    
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
    
            # Mixed precision training
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                # Use the selected criterion instead of hardcoded CrossEntropyLoss
                loss = criterion(logits, labels)
                loss = loss / gradient_accumulation_steps
            
            # Replace loss.backward() with:
            scaler.scale(loss).backward()
            
            # Replace optimizer.step() with:
            if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
    
        # Average training loss
        train_loss /= len(train_loader)
    
        # Validation loop
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []
    
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
    
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                # Use the selected criterion for validation loss too
                loss = criterion(logits, labels)
    
                val_loss += loss.item()
                probabilities = F.softmax(logits, dim=1)
                val_preds.append(probabilities.cpu().numpy())
                val_labels.append(labels.cpu().numpy())
    
        # Calculate validation metrics
        # Calculate validation metrics
        val_preds = np.concatenate(val_preds)
        val_labels = np.concatenate(val_labels)
        val_loss /= len(val_loader)
        
        # Ensure probabilities sum to 1
        if not np.allclose(val_preds.sum(axis=1), 1.0, atol=1e-6):
            print("Warning: val_preds do not sum to 1. Fixing with normalization.")
            val_preds = val_preds / val_preds.sum(axis=1, keepdims=True)
        
        val_preds_class = np.argmax(val_preds, axis=1)
        accuracy = accuracy_score(val_labels, val_preds_class)
        recall = recall_score(val_labels, val_preds_class)
        precision = precision_score(val_labels, val_preds_class)
        f1 = f1_score(val_labels, val_preds_class)
        roc_auc = roc_auc_score(val_labels, val_preds[:, 1])
        pr_auc = average_precision_score(val_labels, val_preds[:, 1])  # Add this line
        
        print(f"Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")
        print(f"Accuracy: {accuracy:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1:.4f}, ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}")
    
        # Early stopping logic
        if roc_auc - best_roc_auc < min_delta:
            early_stopping_count += 1
            print(f"EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}")
            if early_stopping_count >= early_stopping_patience:
                print("Early stopping")
                break
        else:
            best_roc_auc = roc_auc
            early_stopping_count = 0

    # Test evaluation
    model.eval()
    test_loss = 0
    test_preds = []
    test_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
    
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
    
            # Use the selected criterion for test loss
            loss = criterion(logits, labels)
            test_loss += loss.item()
    
            test_preds.append(F.softmax(logits, dim=1).cpu().numpy())
            test_labels.append(labels.cpu().numpy())
    
    # Calculate test metrics
    test_preds = np.concatenate(test_preds)
    test_labels = np.concatenate(test_labels)
    test_loss /= len(test_loader)
    
    test_preds_class = np.argmax(test_preds, axis=1)
    accuracy = accuracy_score(test_labels, test_preds_class)
    recall = recall_score(test_labels, test_preds_class)
    precision = precision_score(test_labels, test_preds_class)
    f1 = f1_score(test_labels, test_preds_class)
    roc_auc = roc_auc_score(test_labels, test_preds[:, 1])
    pr_auc = average_precision_score(test_labels, test_preds[:, 1])  # Add this line
    
    print(f"\nTest Results:")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Accuracy: {accuracy:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}")  # Add PR AUC here
    
    # Add classification report
    print(f"\nClassification Report:")
    print(classification_report(test_labels, test_preds_class, target_names=['Class 0', 'Class 1']))
    
    return {
    'test_accuracy': accuracy,
    'test_recall': recall,
    'test_precision': precision,
    'test_f1': f1,
    'test_roc_auc': roc_auc,
    'test_pr_auc': pr_auc,  # Add this line
    'best_val_roc_auc': best_roc_auc
}

# Dataset configuration
Mp = {
    'google_1000': [
        4e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google val.csv'
    ],
    'opus_1000': [
        3e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus val.csv'
    ],
    'banglanmt_1000': [
        4e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) test MP.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) Train MP.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) val MP.csv'
    ],
    'banglanmt_full': [
        5e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Val.csv'
    ],
    'chatgpt_1000': [
        5e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances val.csv'
    ],
    'opus_full': [
        2e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Test All__.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Train All.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Val All.csv'
    ]
}


# Main execution loop
if __name__ == "__main__":
    models = [
        'FacebookAI/xlm-roberta-base',
        'google-bert/bert-base-multilingual-uncased',
        'csebuetnlp/banglishbert', 
        'sagorsarker/bangla-bert-base'
    ]


    
    models=[models[3]]
    seeds = [42]     # 42] #1, 200,
    results = []

    Mp=Mp['banglanmt_full']
    
    for seed in seeds:
        for model_name in models:
            for key in Mp:
                lr = Mp[0]
                test_loc = Mp[1]
                train_loc = Mp[2]
                val_loc = Mp[3]
                
                print(f"\n{'='*50}")
                print(f"Running: Seed={seed}, Model={model_name}, Dataset={key}")
                print(f"Learning Rate: {lr}")
                print(f"{'='*50}")
                
                try:
                    result = run(seed, model_name, lr, train_loc, val_loc, test_loc,use_focal_loss = False,
                    use_class_balanced_loss = True,
                    use_cost_sensitive_weighting = False)
                    result['seed'] = seed
                    result['model'] = model_name
                    result['dataset'] = key
                    result['learning_rate'] = lr
                    results.append(result)
                except Exception as e:
                    print(f"Error running {model_name} on {key} with seed {seed}: {str(e)}")
                    continue
    
    # Save results to CSV
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv('experiment_results.csv', index=False)
        print(f"\nExperiment completed. Results saved to experiment_results.csv")
        print(f"Total successful runs: {len(results)}")


# New

In [ ]:
# Import all required libraries at the top
from sklearn.metrics import average_precision_score, classification_report
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import os
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig, 
    AutoModelForSequenceClassification, 
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score
from tqdm import tqdm
from normalizer import normalize

# Add these imports at the top of your existing imports
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

# Add these custom loss classes after your imports and before the run function
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class ClassBalancedLoss(nn.Module):
    def __init__(self, samples_per_class, beta=0.9999, reduction='mean'):
        super(ClassBalancedLoss, self).__init__()
        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / np.array(effective_num)
        weights = weights / np.sum(weights) * len(weights)
        self.weights = torch.tensor(weights, dtype=torch.float32)
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        weights = self.weights.to(inputs.device)
        ce_loss = F.cross_entropy(inputs, targets, weight=weights, reduction=self.reduction)
        return ce_loss

def run(seed, model_name, lr, train_loc, val_loc, test_loc, 
        use_focal_loss=False, use_class_balanced_loss=False, use_cost_sensitive_weighting=False,
        focal_alpha=1.0, focal_gamma=2.0, cb_beta=0.9999):
    # Set seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Load data
    train = pd.read_csv(train_loc)
    test = pd.read_csv(test_loc)
    val = pd.read_csv(val_loc)
    
    # Hyperparameters
    batch_size = 16
    max_tokens = 512
    epochs = 200
    best_roc_auc = 0.0
    min_delta = 0.0001
    early_stopping_count = 0
    early_stopping_patience = 3
    gradient_accumulation_steps = 2
    weight_decay = 0.01
    pruning_ratio = 0.3
    dropout_prob = 0.2
    
    # Create directory for saving models
    model_save_dir = f"best_models_{seed}_{model_name.replace('/', '_')}"
    os.makedirs(model_save_dir, exist_ok=True)
    best_model_path = os.path.join(model_save_dir, "best_model")
    
    # Load model configuration and tokenizer
    config = AutoConfig.from_pretrained(model_name, num_labels=2)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Handle tokenizers without pad_token (like XLM-RoBERTa)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load the pre-trained model with the specified configuration
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    class LosDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels
    
        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx])
            return item
    
        def __len__(self):
            return len(self.labels)

    # Normalize text data
    train['text'] = train['text'].apply(normalize)
    val['text'] = val['text'].apply(normalize)
    test['text'] = test['text'].apply(normalize)

    train_labels = train['los_label'].values
    unique_classes = np.unique(train_labels)
    samples_per_class = np.bincount(train_labels)
    
    # Compute class weights for cost-sensitive learning
    class_weights = compute_class_weight('balanced', classes=unique_classes, y=train_labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
    
    # Initialize loss functions
    if use_focal_loss:
        criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
        print(f"Using Focal Loss with alpha={focal_alpha}, gamma={focal_gamma}")
    elif use_class_balanced_loss:
        criterion = ClassBalancedLoss(samples_per_class, beta=cb_beta)
        print(f"Using Class Balanced Loss with beta={cb_beta}")
    elif use_cost_sensitive_weighting:
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
        print(f"Using Cost Sensitive Weighting with weights: {class_weights}")
    else:
        criterion = nn.CrossEntropyLoss()
        print("Using standard Cross Entropy Loss")

    criterion = criterion.to(device)

    # Tokenize data
    train_encodings = tokenizer(train['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)
    val_encodings = tokenizer(val['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)
    test_encodings = tokenizer(test['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)

    # Create datasets
    train_dataset = LosDataset(train_encodings, train['los_label'].tolist())
    val_dataset = LosDataset(val_encodings, val['los_label'].tolist())
    test_dataset = LosDataset(test_encodings, test['los_label'].tolist())

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    # Setup optimizer and scheduler
    warmup_ratio = 0.1
    total_training_steps = (epochs * len(train_loader)) // gradient_accumulation_steps
    num_warmup_steps = int(warmup_ratio * total_training_steps)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_training_steps
    )

    scaler = torch.cuda.amp.GradScaler()
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0
    
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch+1}")):
            if step % gradient_accumulation_steps == 0:
                optimizer.zero_grad()
    
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
    
            # Mixed precision training
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                # Use the selected criterion instead of hardcoded CrossEntropyLoss
                loss = criterion(logits, labels)
                loss = loss / gradient_accumulation_steps
            
            # Replace loss.backward() with:
            scaler.scale(loss).backward()
            
            # Replace optimizer.step() with:
            if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
    
        # Average training loss
        train_loss /= len(train_loader)
    
        # Validation loop
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []
    
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
    
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                # Use the selected criterion for validation loss too
                loss = criterion(logits, labels)
    
                val_loss += loss.item()
                probabilities = F.softmax(logits, dim=1)
                val_preds.append(probabilities.cpu().numpy())
                val_labels.append(labels.cpu().numpy())
    
        # Calculate validation metrics
        val_preds = np.concatenate(val_preds)
        val_labels = np.concatenate(val_labels)
        val_loss /= len(val_loader)
        
        # Ensure probabilities sum to 1
        if not np.allclose(val_preds.sum(axis=1), 1.0, atol=1e-6):
            print("Warning: val_preds do not sum to 1. Fixing with normalization.")
            val_preds = val_preds / val_preds.sum(axis=1, keepdims=True)
        
        val_preds_class = np.argmax(val_preds, axis=1)
        accuracy = accuracy_score(val_labels, val_preds_class)
        recall = recall_score(val_labels, val_preds_class)
        precision = precision_score(val_labels, val_preds_class)
        f1 = f1_score(val_labels, val_preds_class)
        roc_auc = roc_auc_score(val_labels, val_preds[:, 1])
        pr_auc = average_precision_score(val_labels, val_preds[:, 1])
        
        print(f"Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")
        print(f"Accuracy: {accuracy:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1:.4f}, ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}")
    
        # Early stopping logic and model saving
        if roc_auc > best_roc_auc + min_delta:
            best_roc_auc = roc_auc
            early_stopping_count = 0
            # Save the best model
            print(f"New best model found! ROC AUC: {roc_auc:.4f}. Saving model...")
            model.save_pretrained(best_model_path)
            tokenizer.save_pretrained(best_model_path)
            # Also save optimizer and scheduler states for potential resuming
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_roc_auc': best_roc_auc,
                'scaler_state_dict': scaler.state_dict(),
            }, os.path.join(best_model_path, 'training_state.pt'))
            print(f"Model saved to {best_model_path}")
        else:
            early_stopping_count += 1
            print(f"EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}")
            if early_stopping_count >= early_stopping_patience:
                print("Early stopping")
                break

    # Load the best model for test evaluation
    print(f"\nLoading best model from {best_model_path} for test evaluation...")
    try:
        # Load the best model
        best_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)
        best_model = best_model.to(device)
        print(f"Best model loaded successfully. Best validation ROC AUC: {best_roc_auc:.4f}")
    except Exception as e:
        print(f"Error loading best model: {e}")
        print("Using current model for test evaluation...")
        best_model = model

    # Test evaluation using the best model
    best_model.eval()
    test_loss = 0
    test_preds = []
    test_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing with Best Model"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
    
            outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
    
            # Use the selected criterion for test loss
            loss = criterion(logits, labels)
            test_loss += loss.item()
    
            test_preds.append(F.softmax(logits, dim=1).cpu().numpy())
            test_labels.append(labels.cpu().numpy())
    
    # Calculate test metrics
    test_preds = np.concatenate(test_preds)
    test_labels = np.concatenate(test_labels)
    test_loss /= len(test_loader)
    
    test_preds_class = np.argmax(test_preds, axis=1)
    accuracy = accuracy_score(test_labels, test_preds_class)
    recall = recall_score(test_labels, test_preds_class)
    precision = precision_score(test_labels, test_preds_class)
    f1 = f1_score(test_labels, test_preds_class)
    roc_auc = roc_auc_score(test_labels, test_preds[:, 1])
    pr_auc = average_precision_score(test_labels, test_preds[:, 1])
    
    print(f"\nTest Results (using best model):")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Accuracy: {accuracy:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}")
    
    # Add classification report
    print(f"\nClassification Report:")
    print(classification_report(test_labels, test_preds_class, target_names=['Class 0', 'Class 1']))
    
    # Clean up GPU memory
    del best_model
    torch.cuda.empty_cache()
    
    return {
        'test_accuracy': accuracy,
        'test_recall': recall,
        'test_precision': precision,
        'test_f1': f1,
        'test_roc_auc': roc_auc,
        'test_pr_auc': pr_auc,
        'best_val_roc_auc': best_roc_auc,
        'best_model_path': best_model_path
    }

# Dataset configuration
Mp = {
    'google_1000': [
        4e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google val.csv'
    ],
    'opus_1000': [
        3e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus val.csv'
    ],
    'banglanmt_1000': [
        4e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) test MP.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) Train MP.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) val MP.csv'
    ],
    'banglanmt_full': [
        5e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Val.csv'
    ],
    'chatgpt_1000': [
        5e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances val.csv'
    ],
    'opus_full': [
        2e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Test All__.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Train All.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Val All.csv'
    ]
}

# Main execution loop
if __name__ == "__main__":
    models = [
        'FacebookAI/xlm-roberta-base',
        'google-bert/bert-base-multilingual-uncased',
        'csebuetnlp/banglishbert', 
        'sagorsarker/bangla-bert-base'
    ]

    models = [models[3]]
    seeds = [42]
    results = []

    Mp_dataset = Mp['banglanmt_full']
    
    for seed in seeds:
        for model_name in models:
            lr = Mp_dataset[0]
            test_loc = Mp_dataset[1]
            train_loc = Mp_dataset[2]
            val_loc = Mp_dataset[3]
            
            print(f"\n{'='*50}")
            print(f"Running: Seed={seed}, Model={model_name}, Dataset=banglanmt_full")
            print(f"Learning Rate: {lr}")
            print(f"{'='*50}")
            
            try:
                result = run(seed, model_name, lr, train_loc, val_loc, test_loc,
                           use_focal_loss=True,
                           use_class_balanced_loss=False,
                           use_cost_sensitive_weighting=False)
                result['seed'] = seed
                result['model'] = model_name
                result['dataset'] = 'banglanmt_full'
                result['learning_rate'] = lr
                results.append(result)
            except Exception as e:
                print(f"Error running {model_name} on banglanmt_full with seed {seed}: {str(e)}")
                continue
    
    # Save results to CSV
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv('experiment_results.csv', index=False)
        print(f"\nExperiment completed. Results saved to experiment_results.csv")
        print(f"Total successful runs: {len(results)}")
        
        # Print summary of results
        print(f"\nResults Summary:")
        for result in results:
            print(f"Model: {result['model']}")
            print(f"Best Validation ROC AUC: {result['best_val_roc_auc']:.4f}")
            print(f"Test ROC AUC: {result['test_roc_auc']:.4f}")
            print(f"Best Model Path: {result['best_model_path']}")
            print("-" * 30)